In [11]:
from openai import OpenAI
!chcp 65001

Active code page: 65001


In [12]:
ls

 Volume in drive C has no label.
 Volume Serial Number is EC36-8D2C

 Directory of c:\sesac_mingyo\deep_learning\2015_12_15_openAi

2025-12-15  오후 09:43    <DIR>          .
2025-12-15  오후 05:53    <DIR>          ..
2025-12-15  오후 09:55    <DIR>          __pycache__
2025-12-15  오후 09:59            22,356 LLM1.ipynb
2025-12-15  오후 09:49                58 main.py
2025-12-15  오후 09:58               648 module.py
               3 File(s)         23,062 bytes
               3 Dir(s)  401,622,011,904 bytes free


In [16]:

from dotenv import load_dotenv
load_dotenv('C:/Users/USER/Desktop/apikeys.txt')

True

In [17]:
client = OpenAI()

In [18]:
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': '안녕하세요! 저는 곤이라고 합니다.'},
    {'role': 'assistant', 'content': '안녕하세요, 곤 님! 만나서 반갑습니다. 오늘은 어떤 이야기를 나눌까요?'},
    {'role': 'user', 'content': '제 이름을 아시나요?'} 
]

In [ ]:
response = client.chat.completions.create(model='gpt-4o', messages=messages, stream=True)

In [20]:
response = list(response) # stream=True로 응답을 받으면 response는 iterable하여 한번 불러들이면 사라진다. 그래서 따로 한번 저장해줘야함.

In [22]:
for chunk in response:
    print(chunk)
    # if chunk.choices[0].delta.content:
    #     print(chunk.choices[0].delta.content.replace("\n", ""), end="")

ChatCompletionChunk(id='chatcmpl-Cn2TaY39isgqmOc3KCztddVwo9OIP', choices=[Choice(delta=ChoiceDelta(content='', function_call=None, refusal=None, role='assistant', tool_calls=None), finish_reason=None, index=0, logprobs=None)], created=1765803626, model='gpt-4o-2024-08-06', object='chat.completion.chunk', service_tier='default', system_fingerprint='fp_83554c687e', usage=None, obfuscation='ImXiHfaNORi')
ChatCompletionChunk(id='chatcmpl-Cn2TaY39isgqmOc3KCztddVwo9OIP', choices=[Choice(delta=ChoiceDelta(content='네', function_call=None, refusal=None, role=None, tool_calls=None), finish_reason=None, index=0, logprobs=None)], created=1765803626, model='gpt-4o-2024-08-06', object='chat.completion.chunk', service_tier='default', system_fingerprint='fp_83554c687e', usage=None, obfuscation='eHwx8l0MRR4I')
ChatCompletionChunk(id='chatcmpl-Cn2TaY39isgqmOc3KCztddVwo9OIP', choices=[Choice(delta=ChoiceDelta(content=',', function_call=None, refusal=None, role=None, tool_calls=None), finish_reason=None, 

In [ ]:
messages=[
    {
        'role': 'system',
        'content': '인물 목록을 다음 JSON 형식으로 출력해 주세요.\n{\'people\': [\'aaa\', \'bbb\']}'
    },
    {
        'role': 'user',
        'content': '옛날 옛적에 할아버지와 할머니가 살고 있었습니다.'
    }
]

In [ ]:
response = client.chat.completions.create(
    model='gpt-4o', messages=messages, response_format={'type':'json_object'}
)

In [ ]:
print(response.choices[0].message.content)

{"people": ["할아버지", "할머니"]}


In [ ]:
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

In [ ]:
messages = [
    {'role':'user', 'content':[
        {'type':'text', 'text':'이미지를 설명해 주세요.'},
        {'type':'image_url', 'image_url':{'url':image_url}}
        ]}
]

In [ ]:
response = client.chat.completions.create(
    model='gpt-4o', messages=messages
)
print(response.choices[0].message.content)

이미지에는 두 마리의 고양이가 밝은 분홍색 소파에 누워 있는 모습이 보입니다. 고양이들은 편안하게 누워 있으며, 그 사이에는 두 개의 리모컨이 놓여 있습니다. 고양이들의 털 색은 모두 갈색과 검은색 줄무늬가 섞여 있어 보입니다. 전체적으로 평화롭고 아늑한 분위기를 느낄 수 있습니다.


In [ ]:
import json


def get_current_weather(location, unit="Celsius"):
    if "seoul" in location.lower():
        return json.dumps({"location": "Seoul", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "72", "unit": unit}
        )
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

In [ ]:
get_current_weather('Seoul')

'{"location": "Seoul", "temperature": "10", "unit": "Celsius"}'

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
client= OpenAI()

In [ ]:
messages = [{'role':'user', 'content':'서울 날씨는 어떤가요?'}]

In [ ]:
resp = client.chat.completions.create(model='gpt-4o', messages=messages, tools=tools)

In [ ]:
resp_message = resp.choices[0].message

In [ ]:
resp_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_KKYoi98YLevD3KXbL7QmTznV', function=Function(arguments='{"location":"서울, 대한민국","unit":"celsius"}', name='get_current_weather'), type='function')])

In [ ]:
resp_message.to_dict()

{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'tool_calls': [{'id': 'call_KKYoi98YLevD3KXbL7QmTznV',
   'function': {'arguments': '{"location":"서울, 대한민국","unit":"celsius"}',
    'name': 'get_current_weather'},
   'type': 'function'}]}

In [ ]:
messages.append(resp_message.to_dict())

In [ ]:
available_functions = {
    "get_current_weather": get_current_weather,
}

#사용하고 싶은 함수는 여러 개일 수 있으므로 반복문 사용
for tool_call in resp_message.tool_calls:
    # 함수를 실행
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(
        location=function_args.get("location"),
        unit=function_args.get("unit"),
    )
    print(function_response)

    # 함수 실행 결과를 대화 이력으로 messages에 추가
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

{"location": "\uc11c\uc6b8, \ub300\ud55c\ubbfc\uad6d", "temperature": "unknown"}


In [ ]:
messages

[{'role': 'user', 'content': '서울 날씨는 어떤가요?'},
 {'content': None,
  'refusal': None,
  'role': 'assistant',
  'annotations': [],
  'tool_calls': [{'id': 'call_KKYoi98YLevD3KXbL7QmTznV',
    'function': {'arguments': '{"location":"서울, 대한민국","unit":"celsius"}',
     'name': 'get_current_weather'},
    'type': 'function'}]},
 {'tool_call_id': 'call_KKYoi98YLevD3KXbL7QmTznV',
  'role': 'tool',
  'name': 'get_current_weather',
  'content': '{"location": "\\uc11c\\uc6b8, \\ub300\\ud55c\\ubbfc\\uad6d", "temperature": "unknown"}'}]

In [ ]:
second_response = client.chat.completions.create(model='gpt-4o', messages=messages)

In [ ]:
second_response.choices[0].message.content

'현재 서울의 날씨 정보를 바로 확인할 수 없습니다. 최신 날씨 정보를 확인하려면 기상청 웹사이트나 날씨 앱을 참조해 주세요.'

In [ ]:
from langchain.tools import tool 

@tool
def multiply(a: int, b: int) -> int : 
  """두 숫자를 곱합니다."""
  return a * b

@tool 
def get_weather(city: str) -> str :
  """도시의 날씨를 반환하는 함수"""
  return f"{city}의현재 날씨는 맑음입니다."

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = "필요하면 계산 도구를 사용하세요."

agent = create_agent(model=llm, tools=[multiply], system_prompt=prompt)

inputs = {"messages": [{"role": "user", "content": "6 곱하기 9는 얼마야?"}]}

result = agent.invoke(inputs)

print(result)

{'messages': [HumanMessage(content='6 곱하기 9는 얼마야?', additional_kwargs={}, response_metadata={}, id='85325aa8-f5fc-4ad9-98e2-1d084ba4f453'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 67, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa07c96156', 'id': 'chatcmpl-Cn0NPs6WhaGVzuAVCqtNmgoYWL6ht', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b219d-e7b6-79e1-b847-52272e9fdf5b-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6, 'b': 9}, 'id': 'call_4wa7uGHK93cZnUkBmTSMzolB', 'type': 'tool_call'}], usage_metadata={'input_tokens': 67, 'output_tokens': 17, 'total_tokens': 84, 'input_toke

In [ ]:
result['messages'][-1].content

'6 곱하기 9는 54입니다.'